# Etapa 2: Pipeline de Features
Este notebook implementa a função de transformação unificada e gera o dataset pré-processado para a Etapa 3.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Carregamento dos Dados Brutos

In [2]:
DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Função de Pré-processamento Unificada
Esta função encapsula as transformações aprovadas das Etapas 1 e 2 para que possam ser reproduzidas no Streamlit (deploy).

In [3]:
def preprocess_features(df):
    """
    Applies all feature engineering rules. 
    Receives raw dataframe and returns processed dataframe ready for modeling.
    """
    df_proc = df.copy()
    
    # --- 1. Tratamentos da Etapa 1 ---
    # Imputar TotalCharges e converter para numérico
    df_proc["TotalCharges"] = pd.to_numeric(df_proc["TotalCharges"].replace(" ", "0"))
    
    # Criar TicketMedio
    df_proc["TicketMedio"] = df_proc.apply(lambda row: row["TotalCharges"] / row["tenure"] if row["tenure"] > 0 else 0.0, axis=1)
    
    # Dropar TotalCharges original
    df_proc.drop(columns=["TotalCharges"], inplace=True)
    
    # --- 2. Encoding das Variáveis (Etapa 2) ---
    # Target
    if "Churn" in df_proc.columns:
        df_proc["Churn"] = df_proc["Churn"].map({"Yes": 1, "No": 0})
        
    # Binárias Simples
    df_proc["gender"] = df_proc["gender"].map({"Male": 1, "Female": 0})
    
    bin_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
    for col in bin_cols:
        df_proc[col] = df_proc[col].map({"Yes": 1, "No": 0})
        
    # Binárias com "No internet service" / "No phone service" colapsadas para 0
    internet_features = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
    for col in internet_features:
        df_proc[col] = df_proc[col].map({"Yes": 1, "No": 0, "No internet service": 0})
        
    df_proc["MultipleLines"] = df_proc["MultipleLines"].map({"Yes": 1, "No": 0, "No phone service": 0})
    
    # Ordinal Encoding para Contract
    df_proc["Contract"] = df_proc["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})
    
    # One-Hot Encoding para InternetService e PaymentMethod
    df_proc = pd.get_dummies(df_proc, columns=["InternetService", "PaymentMethod"], drop_first=True, dtype=int)
    
    # Dropar customerID
    if "customerID" in df_proc.columns:
        df_proc.drop(columns=["customerID"], inplace=True)
        
    # --- 3. Engenharia de Features ---
    # NumServicos: Soma das features binárias de serviços
    servicos_cols = internet_features + ["PhoneService", "MultipleLines"]
    df_proc["NumServicos"] = df_proc[servicos_cols].sum(axis=1)
    
    return df_proc


## 3. Execução do Pipeline e Salvamento OArtefato

In [4]:
df_processed = preprocess_features(df_raw)

# Verificar shape e dados
print("Shape final:", df_processed.shape)
print("Colunas:", df_processed.columns.tolist())
df_processed.head()

Shape final: (7043, 24)
Colunas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'MonthlyCharges', 'Churn', 'TicketMedio', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'NumServicos']


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,Churn,TicketMedio,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,NumServicos
0,0,0,1,0,1,0,0,0,1,0,...,1,29.85,0,29.850000,0,0,0,1,0,1
1,1,0,0,0,34,1,0,1,0,1,...,0,56.95,0,55.573529,0,0,0,0,1,3
2,1,0,0,0,2,1,0,1,1,0,...,1,53.85,1,54.075000,0,0,0,0,1,3
3,1,0,0,0,45,0,0,1,0,1,...,0,42.30,0,40.905556,0,0,0,0,0,3
4,0,0,0,0,2,1,0,0,0,0,...,1,70.70,1,75.825000,1,0,0,1,0,1


In [5]:
# Salvar artefato
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "telco_churn_features.csv"

df_processed.to_csv(out_path, index=False)
print(f"Dataset processado salvo em: {out_path}")

Dataset processado salvo em: ..\data\processed\telco_churn_features.csv
